<a href="https://colab.research.google.com/github/JozefSL/pyNotes/blob/main/numpy/RegMProd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
from IPython.display import display, HTML

# The generate_arps_profile function is not used for the new well production calculation.

# Define the start date as a variable
START_DATE_STR = '2024-12-01' # This represents the M0 month, e.g., 'YYYY-MM-DD'
timeline_length = 24 # Number of months to forecast



# 1. Load the CSV files
df = pd.read_csv("CurveData.csv")
factor_df = pd.read_csv("FactorDataT.csv")
well_count_df = pd.read_csv("WellCount.csv")

def generate_base_schedule_matrix(type_curve, num_months):
    """
    Generates a reusable base schedule matrix where each row represents a well vintage
    (the month a batch of wells is turned online) and each column represents a calendar timeline month.

    Parameters:
    type_curve (np.ndarray): The absolute production profile vector for a single standard well.
    timeline_months (int): Total number of months to model in the forecast timeline.

    Returns:
    np.ndarray: A 2D matrix of shape (timeline_months, timeline_months) filled with shifted type curves.
    """
    base_matrix = np.zeros((num_months, num_months))

    for start_month in range(num_months):
        # Calculate how many months are left in the forecast timeline from this start month forward
        months_remaining = num_months - start_month

        # Determine how much of the type curve fits into the remaining timeline window
        curve_length = min(len(type_curve), months_remaining)

        # Populate the row starting from the diagonal (the month the well goes online)
        base_matrix[start_month, start_month:start_month + curve_length] = type_curve[:curve_length]

    return base_matrix



# Identify our timeline_length in months for the new decline curve (M1 through Mxx)
# Corrected to use uppercase 'M' for column names to match the CSV data
month_cols = [f'M{i+1}' for i in range(timeline_length)]
num_months = len(month_cols)

# Dictionary to hold the final forecast timelines for analysis
regional_forecasts = {}

# 2. Process data dynamically by Region for new well production
for region, group in df.groupby('Region'):
    # Extract the new well decline curve (normalized) for the current region
    # Convert non-numeric values to 0 and default to zeros if 'newWellDC' is not found
    new_well_decline_curve_values = np.zeros(num_months)
    legacy_decline_curve_values = np.zeros(num_months)
    new_well_decline_curve_row = group.loc[group['Name'] == 'newWellDC', month_cols]
    legacy_decline_curve_row = group.loc[group['Name'] == 'legacyDC', month_cols]
    if not new_well_decline_curve_row.empty:
        new_well_decline_curve_values = pd.to_numeric(new_well_decline_curve_row.iloc[0], errors='coerce').fillna(0).values
    if not legacy_decline_curve_row.empty:
        legacy_decline_curve_values = pd.to_numeric(legacy_decline_curve_row.iloc[0], errors='coerce').fillna(0).values

    # Get IP rate for the current region from FactorDataT.csv
    # Default to 0.0 if not found for the region
    ip_rate = 0.0
    ip_rate_row = factor_df.loc[factor_df['Region'] == region, 'P1']
    if not ip_rate_row.empty:
        ip_rate = ip_rate_row.values[0]
    # Adjust normalized new-well decline values with IP rate for the current region
    new_well_decline_curve_values = new_well_decline_curve_values * ip_rate

    # Get Well Count *vector* for the current region from WellCount.csv
    # This should be a vector of well counts for M1 to M24, where each element corresponds
    # to the well count for the vintage starting in that month.
    well_count_vector = np.zeros(num_months)
    well_count_row_full = well_count_df.loc[(well_count_df['Region'] == region) & (well_count_df['Name'] == 'wellCount'), month_cols]
    if not well_count_row_full.empty:
        well_count_vector = pd.to_numeric(well_count_row_full.iloc[0], errors='coerce').fillna(0).values

    # Generate the reusable Base Schedule Matrix template (completely independent of well counts)
    base_prod_matrix = generate_base_schedule_matrix(new_well_decline_curve_values, num_months)

    # Scale the base unit matrix by the scenario's well count column (well_count_vector) using broadcasting
    new_well_production_matrix = base_prod_matrix * well_count_vector[:, np.newaxis]


    # Get M0 production rate for the current region from FactorDataT.csv
    # Default to 0.0 if not found for the region
    m0_value = 0.0 # Initialize scalar M0
    m0_series_row = factor_df.loc[factor_df['Region'] == region, 'M0']*1000
    if not m0_series_row.empty:
        # Extract scalar value from the series for both calculation and prepending
        m0_value = pd.to_numeric(m0_series_row, errors='coerce').fillna(0).iloc[0]

    # Adjust normalized legacy decline values with M0 rate for the current region
    # Use the scalar m0_value for multiplication
    legacy_decline_curve_values = legacy_decline_curve_values * m0_value

    # Sum columns to get total timeline production for this region (timeline_length months)
    current_regional_forecast = legacy_decline_curve_values + np.sum(new_well_production_matrix, axis=0)

    # Prepend the M0 value and timeline_length months
    regional_forecasts[region] = np.insert(current_regional_forecast, 0, m0_value)





# 3. Combine results into a clean summary DataFrame
# Convert START_DATE_STR to datetime object
start_date_dt = pd.to_datetime(START_DATE_STR)

# Define the new index labels including the dynamic start month
initial_month_label = start_date_dt.strftime('%b_%Y')

# Generate dates for the subsequent months
dates_after_initial_month = pd.date_range(start=start_date_dt + pd.DateOffset(months=1), periods=num_months, freq='MS')
formatted_dates = [d.strftime('%b_%Y') for d in dates_after_initial_month]

new_index_labels = [initial_month_label] + formatted_dates

forecast_df = pd.DataFrame(regional_forecasts, index=new_index_labels)
forecast_df['L48'] = forecast_df.sum(axis=1)
forecast_df = forecast_df.astype(float)

print("Forecast Summary:")

display(forecast_df.style.format("{:,.0f}"))
#print(forecast_df)

Forecast Summary:


,Appalachia,Bakken,EagleFord,Haynesville,Permian,R48,L48
Dec_2024,"37,737,000","1,177,000","1,060,000","16,143,000","6,634,000","1,951,000","64,702,000"
Jan_2025,"37,755,981","1,173,021","1,067,012","16,256,360","6,661,808","1,948,850","64,863,032"
Feb_2025,"37,787,693","1,166,722","1,074,424","16,407,201","6,659,492","1,943,478","65,039,010"
Mar_2025,"37,845,406","1,157,047","1,080,671","16,571,465","6,659,603","1,938,348","65,252,539"
Apr_2025,"37,902,804","1,146,507","1,085,976","16,739,969","6,659,857","1,934,631","65,469,744"
May_2025,"38,026,638","1,138,080","1,090,378","16,910,986","6,661,423","1,930,653","65,758,158"
Jun_2025,"38,098,423","1,131,833","1,095,813","17,081,904","6,670,865","1,928,890","66,007,729"
Jul_2025,"38,162,182","1,128,079","1,101,826","17,257,491","6,683,214","1,927,418","66,260,211"
Aug_2025,"38,256,413","1,125,482","1,108,404","17,433,751","6,693,804","1,925,886","66,543,739"
Sep_2025,"38,319,999","1,127,388","1,113,724","17,567,264","6,703,784","1,924,613","66,756,772"


### Non-Primary Commodity Forecast

Now, let's calculate the non-primary commodity production (e.g., gas for oil regions, oil for gas regions) using the GOR (Gas-Oil Ratio) and GOR growth factors from `FactorDataT.csv`.

In [11]:
# Dictionary to hold non-primary commodity forecasts
non_primary_regional_forecasts = {}

# Define primary commodity types for each region
# Appalachia and Haynesville are gas (Mcf/d), others are oil (bbl/d)
GAS_REGIONS = ['Appalachia', 'Haynesville']

# Iterate through each region to calculate non-primary commodity production
for region, forecast_values in regional_forecasts.items():
    # Get GOR for M0 and gorGrowth for the current region
    # Default to 0 if not found
    gor_m0 = 0.0
    gor_growth = 0.0 # Assuming this is a percentage (e.g., 0.5 for 0.5% growth)

    gor_row = factor_df.loc[factor_df['Region'] == region]
    if not gor_row.empty:
        gor_m0 = pd.to_numeric(gor_row['GOR'], errors='coerce').fillna(0).iloc[0]
        # Assuming gorGrowth is a monthly percentage, so convert to a multiplier
        gor_monthly_growth = pd.to_numeric(gor_row['gorGrowth'], errors='coerce').fillna(0).iloc[0]/timeline_length
    else:
        gor_monthly_growth = 0.0 # No growth if no data

    # Calculate monthly GOR values for the timeline length
    monthly_gor_values = np.zeros(timeline_length+1)
    monthly_gor_values[0] = gor_m0
    for i in range(1, timeline_length+1):
        monthly_gor_values[i] = gor_m0 + i * gor_monthly_growth

    # Derive non-primary production based on the primary commodity
    non_primary_production = np.zeros(timeline_length+1)
    if region in GAS_REGIONS: # Primary is gas, calculate oil
        # Oil (bbl/d) = Gas (Mcf/d) / GOR (Mcf/bbl)
        # Avoid division by zero if GOR is 0
        non_primary_production = np.where(monthly_gor_values != 0, forecast_values / monthly_gor_values, 0)
    else: # Primary is oil, calculate gas
        # Gas (Mcf/d) = Oil (bbl/d) * GOR (Mcf/bbl)
        non_primary_production = forecast_values * monthly_gor_values

    non_primary_regional_forecasts[region] = non_primary_production

# Create a DataFrame for non-primary commodity forecasts
non_primary_forecast_df = pd.DataFrame(non_primary_regional_forecasts, index=new_index_labels)
non_primary_forecast_df['L48'] = non_primary_forecast_df.sum(axis=1)
non_primary_forecast_df = non_primary_forecast_df.astype(float)

print("Primary Commodity Forecast (bbl/day or Mcf/day based on region):")
display(forecast_df.style.format("{:,.0f}"))

print("\nNon-Primary Commodity Forecast (bbl/day or Mcf/day based on region):")
display(non_primary_forecast_df.style.format("{:,.0f}"))

Primary Commodity Forecast (bbl/day or Mcf/day based on region):


,Appalachia,Bakken,EagleFord,Haynesville,Permian,R48,L48
Dec_2024,"37,737,000","1,177,000","1,060,000","16,143,000","6,634,000","1,951,000","64,702,000"
Jan_2025,"37,755,981","1,173,021","1,067,012","16,256,360","6,661,808","1,948,850","64,863,032"
Feb_2025,"37,787,693","1,166,722","1,074,424","16,407,201","6,659,492","1,943,478","65,039,010"
Mar_2025,"37,845,406","1,157,047","1,080,671","16,571,465","6,659,603","1,938,348","65,252,539"
Apr_2025,"37,902,804","1,146,507","1,085,976","16,739,969","6,659,857","1,934,631","65,469,744"
May_2025,"38,026,638","1,138,080","1,090,378","16,910,986","6,661,423","1,930,653","65,758,158"
Jun_2025,"38,098,423","1,131,833","1,095,813","17,081,904","6,670,865","1,928,890","66,007,729"
Jul_2025,"38,162,182","1,128,079","1,101,826","17,257,491","6,683,214","1,927,418","66,260,211"
Aug_2025,"38,256,413","1,125,482","1,108,404","17,433,751","6,693,804","1,925,886","66,543,739"
Sep_2025,"38,319,999","1,127,388","1,113,724","17,567,264","6,703,784","1,924,613","66,756,772"



Non-Primary Commodity Forecast (bbl/day or Mcf/day based on region):


,Appalachia,Bakken,EagleFord,Haynesville,Permian,R48,L48
Dec_2024,"188,685","3,425,070","8,193,800","23,987","29,985,680","26,260,460","68,077,682"
Jan_2025,"191,574","3,428,154","8,279,121","24,006","30,250,161","26,231,524","68,404,540"
Feb_2025,"194,615","3,424,328","8,367,976","24,081","30,378,384","26,159,210","68,548,593"
Mar_2025,"197,884","3,410,395","8,448,146","24,174","30,517,631","26,090,165","68,688,395"
Apr_2025,"201,254","3,393,660","8,521,295","24,273","30,657,543","26,040,129","68,838,153"
May_2025,"205,087","3,382,942","8,587,637","24,373","30,803,529","25,986,590","68,990,158"
Jun_2025,"208,758","3,378,523","8,662,403","24,473","30,986,168","25,962,860","69,223,185"
Jul_2025,"212,504","3,381,417","8,742,071","24,577","31,182,765","25,943,052","69,486,387"
Aug_2025,"216,546","3,387,700","8,826,594","24,682","31,371,629","25,922,420","69,749,572"
Sep_2025,"220,547","3,407,532","8,901,436","24,725","31,558,061","25,905,288","70,017,590"


### Combined Oil and Gas Forecasts

Let's combine the primary and non-primary commodity forecasts into distinct Oil and Gas production dataframes.

In [9]:
oil_forecast_combined = {}
gas_forecast_combined = {}

# Assuming GAS_REGIONS is already defined in a previous cell
# GAS_REGIONS = ['Appalachia', 'Haynesville']

for region in regional_forecasts.keys(): # Iterate through all regions
    if region in GAS_REGIONS: # Primary is Gas, Non-primary is Oil
        gas_forecast_combined[region] = forecast_df[region].values
        oil_forecast_combined[region] = non_primary_forecast_df[region].values
    else: # Primary is Oil, Non-primary is Gas
        oil_forecast_combined[region] = forecast_df[region].values
        gas_forecast_combined[region] = non_primary_forecast_df[region].values

# Create combined Oil and Gas DataFrames
oil_forecast_combined_df = pd.DataFrame(oil_forecast_combined, index=new_index_labels)
gas_forecast_combined_df = pd.DataFrame(gas_forecast_combined, index=new_index_labels)

# Add 'Total Company' (L48) column to both combined dataframes
oil_forecast_combined_df['L48'] = oil_forecast_combined_df.sum(axis=1)
gas_forecast_combined_df['L48'] = gas_forecast_combined_df.sum(axis=1)

# Ensure data types are float
oil_forecast_combined_df = oil_forecast_combined_df.astype(float)
gas_forecast_combined_df = gas_forecast_combined_df.astype(float)

print("\n--- Final Oil Production Forecast (bbl/day) ---")
display(oil_forecast_combined_df.style.format("{:,.0f}"))

print("\n--- Final Gas Production Forecast (Mcf/day) ---")
display(gas_forecast_combined_df.style.format("{:,.0f}"))


--- Final Oil Production Forecast (bbl/day) ---


,Appalachia,Bakken,EagleFord,Haynesville,Permian,R48,L48
Dec_2024,"188,685","1,177,000","1,060,000","23,987","6,634,000","1,951,000","11,034,672"
Jan_2025,"191,574","1,173,021","1,067,012","24,006","6,661,808","1,948,850","11,066,271"
Feb_2025,"194,615","1,166,722","1,074,424","24,081","6,659,492","1,943,478","11,062,812"
Mar_2025,"197,884","1,157,047","1,080,671","24,174","6,659,603","1,938,348","11,057,728"
Apr_2025,"201,254","1,146,507","1,085,976","24,273","6,659,857","1,934,631","11,052,497"
May_2025,"205,087","1,138,080","1,090,378","24,373","6,661,423","1,930,653","11,049,994"
Jun_2025,"208,758","1,131,833","1,095,813","24,473","6,670,865","1,928,890","11,060,633"
Jul_2025,"212,504","1,128,079","1,101,826","24,577","6,683,214","1,927,418","11,077,620"
Aug_2025,"216,546","1,125,482","1,108,404","24,682","6,693,804","1,925,886","11,094,804"
Sep_2025,"220,547","1,127,388","1,113,724","24,725","6,703,784","1,924,613","11,114,781"



--- Final Gas Production Forecast (Mcf/day) ---


,Appalachia,Bakken,EagleFord,Haynesville,Permian,R48,L48
Dec_2024,"37,737,000","3,425,070","8,193,800","16,143,000","29,985,680","26,260,460","121,745,010"
Jan_2025,"37,755,981","3,428,154","8,279,121","16,256,360","30,250,161","26,231,524","122,201,301"
Feb_2025,"37,787,693","3,424,328","8,367,976","16,407,201","30,378,384","26,159,210","122,524,791"
Mar_2025,"37,845,406","3,410,395","8,448,146","16,571,465","30,517,631","26,090,165","122,883,207"
Apr_2025,"37,902,804","3,393,660","8,521,295","16,739,969","30,657,543","26,040,129","123,255,400"
May_2025,"38,026,638","3,382,942","8,587,637","16,910,986","30,803,529","25,986,590","123,698,322"
Jun_2025,"38,098,423","3,378,523","8,662,403","17,081,904","30,986,168","25,962,860","124,170,281"
Jul_2025,"38,162,182","3,381,417","8,742,071","17,257,491","31,182,765","25,943,052","124,668,979"
Aug_2025,"38,256,413","3,387,700","8,826,594","17,433,751","31,371,629","25,922,420","125,198,507"
Sep_2025,"38,319,999","3,407,532","8,901,436","17,567,264","31,558,061","25,905,288","125,659,581"


### GOR Ratios for the Forecasted Window

Let's calculate and display the Gas-Oil Ratios (GOR) for each month in the forecast window, showing how they evolve per region.

In [10]:
gor_ratios_forecast = {}

for region in regional_forecasts.keys():
    gor_m0 = 0.0
    gor_monthly_growth = 0.0

    gor_row = factor_df.loc[factor_df['Region'] == region]
    if not gor_row.empty:
        gor_m0 = pd.to_numeric(gor_row['GOR'], errors='coerce').fillna(0).iloc[0]
        gor_monthly_growth = pd.to_numeric(gor_row['gorGrowth'], errors='coerce').fillna(0).iloc[0] / timeline_length

    monthly_gor_values_for_region = np.zeros(timeline_length + 1)
    monthly_gor_values_for_region[0] = gor_m0
    for i in range(1, timeline_length + 1):
        monthly_gor_values_for_region[i] = gor_m0 + i * gor_monthly_growth

    gor_ratios_forecast[region] = monthly_gor_values_for_region

gor_ratios_df = pd.DataFrame(gor_ratios_forecast, index=new_index_labels)
gor_ratios_df = gor_ratios_df.astype(float)

print("\n--- GOR Ratios (Mcf/bbl) for Forecasted Window ---")
display(gor_ratios_df.style.format("{:,.2f}"))


--- GOR Ratios (Mcf/bbl) for Forecasted Window ---


,Appalachia,Bakken,EagleFord,Haynesville,Permian,R48
Dec_2024,200.00,2.91,7.73,673.00,4.52,13.46
Jan_2025,197.08,2.92,7.76,677.17,4.54,13.46
Feb_2025,194.17,2.94,7.79,681.33,4.56,13.46
Mar_2025,191.25,2.95,7.82,685.50,4.58,13.46
Apr_2025,188.33,2.96,7.85,689.67,4.60,13.46
May_2025,185.42,2.97,7.88,693.83,4.62,13.46
Jun_2025,182.50,2.99,7.91,698.00,4.64,13.46
Jul_2025,179.58,3.00,7.93,702.17,4.67,13.46
Aug_2025,176.67,3.01,7.96,706.33,4.69,13.46
Sep_2025,173.75,3.02,7.99,710.50,4.71,13.46


### Final Marketed Gas Production Forecast

Now, let's adjust the 'Final Gas Production Forecast' to account for the `ShrinkFactor` for each region, deriving the 'Final Marketed Gas Production Forecast'.

In [15]:
marketed_gas_forecast = {}

for region in gas_forecast_combined_df.columns:
    if region == 'L48': # Skip the total column for this calculation, it will be re-calculated
        continue

    shrink_factor = 1.0 # Default shrink factor if not found
    shrink_row_series = factor_df.loc[factor_df['Region'] == region, 'ShrinkFactor']
    if not shrink_row_series.empty:
        # Convert to numeric and fill any NaN values in the Series before extracting the scalar
        shrink_factor = pd.to_numeric(shrink_row_series, errors='coerce').fillna(1.0).iloc[0]

    # Ensure shrink_factor is not zero to prevent division errors
    if shrink_factor == 0:
        shrink_factor = 1.0

    marketed_gas_forecast[region] = gas_forecast_combined_df[region] * shrink_factor

marketed_gas_forecast_df = pd.DataFrame(marketed_gas_forecast, index=new_index_labels)
marketed_gas_forecast_df['L48'] = marketed_gas_forecast_df.sum(axis=1)
marketed_gas_forecast_df = marketed_gas_forecast_df.astype(float)

print("\n--- Final Marketed Gas Production Forecast (Mcf/day) ---")
display(marketed_gas_forecast_df.style.format("{:,.0f}"))


--- Final Marketed Gas Production Forecast (Mcf/day) ---


,Appalachia,Bakken,EagleFord,Haynesville,Permian,R48,L48
Dec_2024,"37,737,000","3,219,566","7,349,839","16,062,285","28,336,468","25,498,907","118,204,064"
Jan_2025,"37,755,981","3,222,465","7,426,371","16,175,078","28,586,402","25,470,810","118,637,108"
Feb_2025,"37,787,693","3,218,868","7,506,074","16,325,165","28,707,572","25,400,593","118,945,966"
Mar_2025,"37,845,406","3,205,771","7,577,987","16,488,607","28,839,161","25,333,550","119,290,482"
Apr_2025,"37,902,804","3,190,041","7,643,601","16,656,269","28,971,378","25,284,966","119,649,058"
May_2025,"38,026,638","3,179,965","7,703,111","16,826,431","29,109,334","25,232,979","120,078,458"
Jun_2025,"38,098,423","3,175,811","7,770,176","16,996,495","29,281,929","25,209,937","120,532,770"
Jul_2025,"38,162,182","3,178,532","7,841,638","17,171,204","29,467,713","25,190,704","121,011,972"
Aug_2025,"38,256,413","3,184,438","7,917,455","17,346,582","29,646,190","25,170,670","121,521,747"
Sep_2025,"38,319,999","3,203,080","7,984,588","17,479,428","29,822,368","25,154,035","121,963,498"


In [7]:
monthly_gor_values
#gor_monthly_growth

array([13.46, 13.46, 13.46, 13.46, 13.46, 13.46, 13.46, 13.46, 13.46,
       13.46, 13.46, 13.46, 13.46, 13.46, 13.46, 13.46, 13.46, 13.46,
       13.46, 13.46, 13.46, 13.46, 13.46, 13.46, 13.46])

In [17]:
new_well_decline_curve_values

array([170.274, 390.   , 371.046, 332.007, 291.564, 255.723, 225.03 ,
       200.694, 177.606, 160.836, 146.445, 134.238, 124.644, 116.493,
       108.732, 101.634,  95.784,  90.597,  85.41 ,  80.34 ,  76.557,
        73.983,  70.707,  67.938])

In [25]:
print('New Well Production Matrix (BOE/day):')
new_well_prod_df = pd.DataFrame(new_well_production_matrix)
display(new_well_prod_df.style.format("{:,.0f}"))
print("\n")

New Well Production Matrix (BOE/day):


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
0,"29,652","67,915","64,614","57,816","50,773","44,532","39,187","34,949","30,928","28,008","25,502","23,376","21,706","20,286","18,935","17,699","16,680","15,777","14,873","13,990","13,332","12,883","12,313","11,831"
1,0,"29,917","68,523","65,193","58,334","51,228","44,931","39,538","35,262","31,205","28,259","25,730","23,586","21,900","20,468","19,104","17,857","16,829","15,918","15,007","14,116","13,451","12,999","12,423"
2,0,0,"30,191","69,151","65,790","58,868","51,697","45,342","39,900","35,585","31,491","28,518","25,966","23,802","22,101","20,655","19,279","18,021","16,983","16,064","15,144","14,245","13,574","13,118"
3,0,0,0,"30,469","69,787","66,395","59,409","52,172","45,759","40,267","35,912","31,781","28,780","26,205","24,021","22,304","20,845","19,457","18,186","17,140","16,211","15,283","14,376","13,699"
4,0,0,0,0,"30,741","70,411","66,989","59,941","52,639","46,168","40,627","36,233","32,065","29,037","26,439","24,235","22,503","21,032","19,630","18,349","17,293","16,356","15,420","14,505"
5,0,0,0,0,0,"31,003","71,011","67,560","60,452","53,088","46,562","40,973","36,542","32,339","29,285","26,665","24,442","22,695","21,211","19,798","18,506","17,440","16,496","15,551"
6,0,0,0,0,0,0,"31,249","71,573","68,094","60,930","53,508","46,930","41,298","36,831","32,594","29,517","26,876","24,635","22,875","21,379","19,954","18,652","17,578","16,626"
7,0,0,0,0,0,0,0,"31,470","72,080","68,577","61,362","53,887","47,263","41,590","37,092","32,825","29,726","27,066","24,810","23,037","21,530","20,096","18,784","17,703"
8,0,0,0,0,0,0,0,0,"31,666","72,528","69,003","61,743","54,222","47,557","41,849","37,323","33,029","29,911","27,234","24,964","23,180","21,664","20,221","18,901"
9,0,0,0,0,0,0,0,0,0,"31,831","72,907","69,363","62,065","54,505","47,805","42,067","37,518","33,202","30,067","27,376","25,094","23,301","21,777","20,326"
